# Partycja per device i day oraz wymuszony kierunek DESC clustering po event_time
Wszystkie dane urządzenia trafiają teraz na partycję złożoną (device_id, day), co sprawia, że odczyt staje się szybszy.
Dane posortowane są w kolejności od najnowszych do najstarszych.
Partition key = device_id, day
Clustering key = event_time DESC

In [253]:
import uuid, time
from cassandra.cluster import Cluster

In [254]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

Połączono z Cassandra


In [255]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

Utworzono keyspace


In [256]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_4 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY ((device_id, day), event_time)
    ) WITH CLUSTERING ORDER BY (event_time DESC)
""")
print("Utworzono tabelę 'users'")

Utworzono tabelę 'users'


## Generate test data

In [257]:
!python ../_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_4 \
  --records 10000 \
  --batch-size 100

Done. Loader=cassandra, Records=10000


In [258]:
rows = session.execute("""
SELECT count(*)
FROM
    events_by_device_4
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"COUNT: {row.count}")

Znalezione rekordy:
COUNT: 118193


### Podstawienie pełnego klucza

In [ ]:
start = time.perf_counter()
rows = session.execute("""
SELECT *
FROM
    events_by_device_4
WHERE
    device_id='device_1'
    AND day='2026-03-29'
LIMIT 10
""")
end = time.perf_counter()
print(f"Zapytanie wykonane w {end - start:.4f} sekund")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

Zapytanie wykonane w 0.0093 sekund
Znalezione rekordy:
- device_1 2026-03-28 (24.809999465942383°C, 65.01000213623047%) - 2026-03-28 16:21:19.075000
- device_1 2026-03-28 (24.459999084472656°C, 51.790000915527344%) - 2026-03-28 16:21:19.069000
- device_1 2026-03-28 (27.549999237060547°C, 55.599998474121094%) - 2026-03-28 16:20:51.946000
- device_1 2026-03-28 (26.209999084472656°C, 50.939998626708984%) - 2026-03-28 16:20:51.912000
- device_1 2026-03-28 (32.47999954223633°C, 38.13999938964844%) - 2026-03-28 16:20:51.867000
- device_1 2026-03-28 (29.68000030517578°C, 46.22999954223633%) - 2026-03-28 16:20:51.808000
- device_1 2026-03-28 (25.6299991607666°C, 37.720001220703125%) - 2026-03-28 16:20:51.675000
- device_1 2026-03-28 (34.619998931884766°C, 46.560001373291016%) - 2026-03-28 16:20:51.631000
- device_1 2026-03-28 (29.149999618530273°C, 62.560001373291016%) - 2026-03-28 16:20:51.626000
- device_1 2026-03-28 (27.770000457763672°C, 55.70000076293945%) - 2026-03-28 16:20:51.483000


### Podstawienie tylko device_id

In [260]:
# rows = session.execute("""
# SELECT *
# FROM
#     events_by_device_4
# WHERE
#     device_id='device_1'
# LIMIT 10
# """)
# print("Znalezione rekordy:")
# for row in rows:
#     print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

### Brak podstawienie partition key

In [261]:
# try:
#     session.execute("""
#     SELECT *
#     FROM
#         events_by_device_4
#     WHERE
#         temperature > 25
#     """)
# except Exception as e:
#     print(e)

### ALLOW FILTERING

In [262]:
# rows = session.execute("""
# SELECT *
# FROM
#     events_by_device_4
# WHERE
#     temperature > 25 ALLOW FILTERING
# """)
# print("Znalezione rekordy:")
# for row in rows:
#     print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

In [263]:
# Zamknięcie połączenia
cluster.shutdown()